In [1]:
import os
import gc
import re
import time
import random
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf

from pathlib import Path
from tensorflow.keras import layers, Model, Input

print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# GPU setup
gpus = tf.config.list_physical_devices("GPU")
print("Available GPUs:", len(gpus))

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

I0000 00:00:1794932563.479827      16 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794932582.279584      16 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow: 2.21.0
OpenCV: 4.14.0
NumPy: 2.1.3
Available GPUs: 0


E0000 00:00:1794932586.990277      16 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
PREPARED = Path(
    "/kaggle/input/datasets/swaratakale/dip-project-prepared-dataset"
)

DATASET = PREPARED / "dataset"

TRAIN_DIR = DATASET / "train" / "Outputs"
VAL_INPUT_DIR = DATASET / "validation" / "Inputs"
VAL_OUTPUT_DIR = DATASET / "validation" / "Outputs"
TEST_INPUT_DIR = DATASET / "test" / "Inputs"
TEST_OUTPUT_DIR = DATASET / "test" / "Outputs"

PROJECT = Path("/kaggle/working/DIP_Project")
CHECKPOINTS = PROJECT / "checkpoints"
RESULTS = PROJECT / "results"
PREDICTIONS = PROJECT / "predictions"

for folder in [CHECKPOINTS, RESULTS, PREDICTIONS]:
    folder.mkdir(parents=True, exist_ok=True)

train_files = sorted(TRAIN_DIR.glob("*"))
val_inputs = sorted(VAL_INPUT_DIR.glob("*"))
val_outputs = sorted(VAL_OUTPUT_DIR.glob("*"))
test_inputs = sorted(TEST_INPUT_DIR.glob("*"))
test_outputs = sorted(TEST_OUTPUT_DIR.glob("*"))

print("Training:", len(train_files))
print("Validation:", len(val_inputs))
print("Testing:", len(test_inputs))

assert len(train_files) == 200
assert len(val_inputs) == len(val_outputs) == 25
assert len(test_inputs) == len(test_outputs) == 25

Training: 200
Validation: 25
Testing: 25


In [3]:
PATCH_SIZE = 352

def make_training_pair(image_path):

    image = cv2.imread(
        str(image_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise ValueError(f"Could not read {image_path}")

    h, w = image.shape

    scale = max(
        PATCH_SIZE / min(h, w),
        random.uniform(0.75, 1.25)
    )

    new_w = max(PATCH_SIZE, int(w * scale))
    new_h = max(PATCH_SIZE, int(h * scale))

    image = cv2.resize(image, (new_w, new_h))

    angle = random.uniform(0, 360)

    matrix = cv2.getRotationMatrix2D(
        (new_w / 2, new_h / 2),
        angle,
        1
    )

    image = cv2.warpAffine(
        image,
        matrix,
        (new_w, new_h),
        borderValue=255
    )

    if random.random() < 0.5:
        image = cv2.flip(image, 1)

    if random.random() < 0.5:
        image = cv2.flip(image, 0)

    h, w = image.shape

    x = random.randint(0, w - PATCH_SIZE)
    y = random.randint(0, h - PATCH_SIZE)

    clean = image[
        y:y+PATCH_SIZE,
        x:x+PATCH_SIZE
    ].copy()

    damaged = clean.copy()

    # Foreground pixels represent black drawing lines
    ys, xs = np.where(clean < 128)

    for _ in range(random.randint(10, 20)):

        side = random.randint(10, 50)

        # Mostly centre gaps on existing drawing lines
        if len(xs) > 0 and random.random() < 0.8:

            idx = random.randrange(len(xs))
            cx, cy = int(xs[idx]), int(ys[idx])

            x1 = cx - side // 2
            y1 = cy - side // 2

        else:
            x1 = random.randint(0, PATCH_SIZE - side)
            y1 = random.randint(0, PATCH_SIZE - side)

        # Keep the entire square inside the patch
        x1 = max(0, min(x1, PATCH_SIZE - side))
        y1 = max(0, min(y1, PATCH_SIZE - side))

        damaged[
            y1:y1+side,
            x1:x1+side
        ] = 255

    clean = clean.astype(np.float32) / 255.0
    damaged = damaged.astype(np.float32) / 255.0

    return damaged[..., None], clean[..., None]


def training_generator(batch_size):

    while True:

        batch_x = []
        batch_y = []

        for _ in range(batch_size):

            image_path = random.choice(train_files)

            damaged, clean = make_training_pair(image_path)

            batch_x.append(damaged)
            batch_y.append(clean)

        yield np.stack(batch_x), np.stack(batch_y)

In [ ]:
import matplotlib.pyplot as plt

damaged, clean = make_training_pair(random.choice(train_files))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(damaged[..., 0], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Damaged Input")

axes[1].imshow(clean[..., 0], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Clean Target")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [5]:
def conv_bn_relu(x, filters, kernel_size=3, strides=1):

    x = layers.Conv2D(
        filters,
        kernel_size,
        strides=strides,
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    return x


def build_model1():

    inputs = Input(shape=(None, None, 1))

    x = conv_bn_relu(inputs, 24, kernel_size=5, strides=2)
    x = conv_bn_relu(x, 64, strides=2)
    x = conv_bn_relu(x, 128)
    x = conv_bn_relu(x, 256, strides=2)
    x = conv_bn_relu(x, 512)
    x = conv_bn_relu(x, 512, strides=2)
    x = conv_bn_relu(x, 256)

    x = layers.UpSampling2D(2, interpolation="nearest")(x)
    x = conv_bn_relu(x, 128)
    x = conv_bn_relu(x, 64)

    x = layers.UpSampling2D(2, interpolation="nearest")(x)
    x = conv_bn_relu(x, 32)
    x = conv_bn_relu(x, 16)

    x = layers.UpSampling2D(2, interpolation="nearest")(x)
    x = conv_bn_relu(x, 8)
    x = conv_bn_relu(x, 4)

    x = layers.UpSampling2D(2, interpolation="nearest")(x)
    x = conv_bn_relu(x, 2)
    x = conv_bn_relu(x, 1)

    outputs = layers.Conv2D(
        1, 3,
        padding="same",
        activation="sigmoid"
    )(x)

    return Model(inputs, outputs, name="Encoder_Decoder")


def conv_bn_relu_dilated(
    x, filters, kernel_size=3, dilation_rate=1
):

    x = layers.Conv2D(
        filters,
        kernel_size,
        padding="same",
        dilation_rate=dilation_rate,
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    return x


def build_model2():

    inputs = Input(shape=(None, None, 1))

    x = conv_bn_relu_dilated(inputs, 64, kernel_size=11)
    x = conv_bn_relu_dilated(x, 128)
    x = conv_bn_relu_dilated(x, 256, dilation_rate=1)
    x = conv_bn_relu_dilated(x, 256, dilation_rate=2)
    x = conv_bn_relu_dilated(x, 256, dilation_rate=4)
    x = conv_bn_relu_dilated(x, 256, dilation_rate=8)
    x = conv_bn_relu_dilated(x, 128)
    x = conv_bn_relu_dilated(x, 64)

    outputs = layers.Conv2D(
        1, 3,
        padding="same",
        activation="sigmoid"
    )(x)

    return Model(inputs, outputs, name="Dilated_CNN")


model1 = build_model1()
model2 = build_model2()

print("Model 1 parameters:", model1.count_params())
print("Model 2 parameters:", model2.count_params())

Model 1 parameters: 5502904
Model 2 parameters: 2520705


In [6]:
def find_latest_checkpoint(model_name):

    files = list(
        CHECKPOINTS.glob(f"{model_name}_iter_*.keras")
    )

    if not files:
        return None, 0

    def get_iteration(path):
        match = re.search(r"iter_(\d+)", path.name)
        return int(match.group(1))

    latest = max(files, key=get_iteration)

    return latest, get_iteration(latest)


class CheckpointCallback(tf.keras.callbacks.Callback):

    def __init__(
        self,
        model_name,
        start_iteration,
        interval=5000
    ):
        super().__init__()

        self.model_name = model_name
        self.start_iteration = start_iteration
        self.interval = interval

    def on_train_batch_end(self, batch, logs=None):

        iteration = self.start_iteration + batch + 1

        if iteration % self.interval == 0:

            path = CHECKPOINTS / (
                f"{self.model_name}_iter_{iteration:08d}.keras"
            )

            self.model.save(path)

            print(f"\nSaved checkpoint: {path.name}")


def train_experiment(
    model_name,
    model_builder,
    batch_size,
    iterations=10000,
    checkpoint_interval=5000
):

    latest, start_iteration = find_latest_checkpoint(model_name)

    if latest:

        print("Resuming from:", latest.name)
        model = tf.keras.models.load_model(latest)

    else:

        print("Starting new experiment:", model_name)

        model = model_builder()

        model.compile(
            optimizer=tf.keras.optimizers.Adadelta(),
            loss="mse"
        )

    print("Starting iteration:", start_iteration + 1)
    print("Iterations to run:", iterations)

    callback = CheckpointCallback(
        model_name,
        start_iteration,
        checkpoint_interval
    )

    start_time = time.time()

    history = model.fit(
        training_generator(batch_size),
        steps_per_epoch=iterations,
        epochs=1,
        callbacks=[callback],
        verbose=1
    )

    total_iterations = start_iteration + iterations

    final_path = CHECKPOINTS / (
        f"{model_name}_iter_{total_iterations:08d}.keras"
    )

    if not final_path.exists():
        model.save(final_path)

    elapsed = (time.time() - start_time) / 60

    print("\nTraining completed!")
    print("Total iterations:", total_iterations)
    print("Final loss:", history.history["loss"][-1])
    print("Time (minutes):", round(elapsed, 2))
    print("Final checkpoint:", final_path)

    return model, total_iterations

In [7]:
# # MODEL 1: Initial training

# tf.keras.backend.clear_session()
# gc.collect()

# model1, model1_iterations = train_experiment(
#     model_name="Model1_linecentered",
#     model_builder=build_model1,
#     batch_size=4,
#     iterations=10000,
#     checkpoint_interval=5000
# )

In [8]:
# # MODEL 1: Extension to 40,000 iterations

# del model1
# tf.keras.backend.clear_session()
# gc.collect()

# model1, model1_iterations = train_experiment(
#     model_name="Model1_linecentered",
#     model_builder=build_model1,
#     batch_size=4,
#     iterations=10000,
#     checkpoint_interval=5000
# )

In [9]:
# tf.keras.backend.clear_session()
# gc.collect()

# model2, model2_iterations = train_experiment(
#     model_name="Model2_linecentered",
#     model_builder=build_model2,
#     batch_size=2,
#     iterations=50000,
#     checkpoint_interval=5000
# )

In [10]:
import os, sys, subprocess, shutil
from pathlib import Path
import cv2, numpy as np, pandas as pd
import tensorflow as tf

BASE = Path("/kaggle/input/datasets/swaratakale")
CBO = BASE / "lineartcompletion-dip-project/CBO_Dataset_20181113/CBO_Dataset/By Item"
FILES = BASE / "project-complete-files-final"

GENERATOR = FILES / "generate_inputs.py"
EVALUATOR = FILES / "evaluate.py"

WORK = Path("/kaggle/working/line_completion_eval")
SOURCE, INPUTS, GT, MASKS = [WORK/x for x in ["Source", "Inputs", "GT", "Masks"]]
PRED, LOGS = WORK/"Predictions", WORK/"Logs"

for p in [SOURCE, INPUTS, GT, MASKS, PRED, LOGS]:
    shutil.rmtree(p, ignore_errors=True)
    p.mkdir(parents=True)

In [11]:
image_files = sorted(
    p for p in CBO.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}
)

for path in image_files:
    name = "__".join(path.relative_to(CBO).with_suffix("").parts) + ".png"
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        cv2.imwrite(str(SOURCE/name), img)

print("Source images:", len(list(SOURCE.glob("*.png"))))

subprocess.run([
    sys.executable, str(GENERATOR),
    str(SOURCE), str(INPUTS),
    "--gt-dir", str(GT),
    "--mask-dir", str(MASKS),
    "--seed", "42"
], check=True)

print("Generated inputs:", len(list(INPUTS.glob("*.png"))))
print("Generated ground truths:", len(list(GT.glob("*.png"))))

Source images: 250
----------------------------------------
Saved: 250 | Skipped (no valid gap site): 0 | Errors: 0
Gaps: 8956 total, 35.8 per image
Output Directory: /kaggle/working/line_completion_eval/Inputs
Generated inputs: 250
Generated ground truths: 250


In [12]:
M1_INC = BASE / "model1-checkpoints-final"
M1_50K = BASE / "trained-checkpoints-50000/Model1_iter_00050000"
M2_50K = BASE / "trained-checkpoints-50000/Model2_iter_00050000"

CHECKPOINTS = {
    f"Model1_{i//1000}k": M1_INC / f"Model1_linecentered_iter_{i:08d}"
    for i in [10000, 20000, 30000, 40000]
}
CHECKPOINTS.update({
    "Model1_50k": M1_50K,
    "Model2_50k": M2_50K
})

for name, path in CHECKPOINTS.items():
    print(name, "OK" if (path/"model.weights.h5").exists() else "MISSING")

Model1_10k OK
Model1_20k OK
Model1_30k OK
Model1_40k OK
Model1_50k OK
Model2_50k OK


In [13]:
def predict_folder(model, input_dir, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)

    for path in sorted(input_dir.glob("*.png")):
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        h, w = img.shape

        ph, pw = (-h) % 16, (-w) % 16
        img = cv2.copyMakeBorder(
            img, 0, ph, 0, pw,
            cv2.BORDER_CONSTANT, value=255
        )

        x = img.astype(np.float32)[None, ..., None] / 255.0
        y = model(x, training=False).numpy()[0, ..., 0]
        y = np.clip(y[:h, :w] * 255, 0, 255).astype(np.uint8)

        cv2.imwrite(str(output_dir/path.name), y)

In [14]:


# for name, path in CHECKPOINTS.items():
#     print(f"\nPredicting: {name}")

#     model = build_model1() if name.startswith("Model1") else build_model2()

#     expected = 5502904 if name.startswith("Model1") else 2520705
#     assert model.count_params() == expected, (
#         f"{name}: parameter mismatch. Got {model.count_params()}, expected {expected}"
#     )

#     model.load_weights(str(path/"model.weights.h5"))

#     predict_folder(model, INPUTS, PRED/name)

#     del model
#     tf.keras.backend.clear_session()

# print("All predictions complete.")

In [15]:

# for name in CHECKPOINTS:
#     print(f"\nEvaluating: {name}")

#     subprocess.run([
#         sys.executable, str(EVALUATOR),
#         str(GT), str(PRED/name), str(INPUTS),
#         "--log-file", str(LOGS/f"{name}.csv")
#     ], check=True)

In [16]:
# metrics = [
#     "psnr", "ssim", "mse", "precision", "recall", "f1",
#     "tol_f1@2", "gap_recall@2", "gap_close_rate",
#     "bridge_precision", "event_f1", "chamfer", "hd95", "cc_error"
# ]

# rows = []

# for name in CHECKPOINTS:
#     df = pd.read_csv(LOGS/f"{name}.csv")
#     row = {"Checkpoint": name, "Images": len(df)}

#     for metric in metrics:
#         col = f"Out_{metric}"
#         if col in df.columns:
#             row[metric] = df[col].mean()

#     rows.append(row)

# comparison = pd.DataFrame(rows)
# comparison.to_csv(LOGS/"model_comparison.csv", index=False)
# display(comparison.round(4))